# In-silico chromatin perturbation of transcription regulators (ATAC → RNA)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ashford-A/UniVI/blob/main/docs/tutorials/experimental/tf_perturbation_multiome.ipynb)

> **Experimental.** This notebook asks a trained model "what RNA would you predict if these peaks were
> closed?". The answer describes what the **model** learned to associate between accessibility and
> expression. It is a hypothesis generator, not a measurement of a causal perturbation, and it has not
> been validated against CRISPR or knockout data. Every effect below is therefore calibrated against
> random peak sets, and the notebook checks whether a second, independently trained model agrees.

The workflow, on 10x Multiome PBMCs:

1. Train a **peak-level** model: ATAC enters as binarized peaks with a Bernoulli decoder, so an edit is
   applied to named genomic regions rather than to LSI coordinates.
2. **Gate** the model: check alignment and ATAC → RNA prediction accuracy before trusting perturbations,
   and see how well each regulator's own expression is predicted.
3. Build peak sets for a panel of hematopoietic transcription regulators in two ways:
   **cis** (peaks around the regulator's own gene) and **motif** (peaks carrying its JASPAR motif).
4. Close each set with `predict_feature_perturbation(mode="off")`, decode RNA, and summarize the change
   per cell type.
5. Compare every effect with **frequency-matched random peak sets** (an empirical null).
6. Look at latent shifts, dose-response, a combination of two regulators, and agreement with the LSI
   model used in the paper.

**Runtime.** On a Colab GPU, training takes a few minutes per model. The optional motif scan
downloads the hg38 genome from UCSC (about 940 MB) and streams through it once, which takes several
minutes; set `RUN_MOTIF_SCAN = False` below to skip it.

In [ ]:
import sys

if "google.colab" in sys.modules:
    %pip install -q "univi[tutorials]>=1.1" "pandas==2.2.3" "pyjaspar>=3"

In [ ]:
import gzip
import os
import re
import urllib.request
from pathlib import Path

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
import torch
from IPython.display import display

import univi
import univi.datasets as uds
from univi import ModalityConfig, TrainingConfig, UniVIConfig, UniVIMultiModalVAE, UniVITrainer
from univi.evaluation import cross_modal_predict, encode_adata, evaluate_alignment, pearson_corr_per_feature
from univi.perturbation import predict_feature_perturbation
from univi.preprocessing import ATACPreprocessor, RNAPreprocessor, split_by_label
from univi.utils.seed import set_seed
from univi.workflows import make_loader

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
set_seed(0)
dense = lambda x: x.toarray() if sp.issparse(x) else np.asarray(x)
print(f"UniVI {univi.__version__} on {device}")

All settings are in one cell. `TF_PANEL` lists regulators with well-described roles in blood cells;
regulators missing from the data are dropped automatically.

In [ ]:
N_EPOCHS = 400            # upper bound; early stopping usually ends sooner
BATCH_SIZE = 256
N_HVG = 2000              # highly variable genes (the panel regulators are always added)
N_PEAKS = 30000           # most frequently open peaks in training cells (cis peaks are always added)
N_LSI = 101               # LSI components for the comparison model
ATAC_RECON_WEIGHT = 0.1   # the Bernoulli term sums over ~15x more features than RNA; tune if needed
TF_PANEL = ["SPI1", "CEBPA", "CEBPB", "IRF8", "PAX5", "EBF1", "TCF7", "LEF1", "TBX21", "EOMES"]
CIS_WINDOW_BP = 50_000    # peaks within this distance of the regulator's TSS form its cis set
N_NULL = 50               # random peak sets per perturbation for the empirical null
RUN_MOTIF_SCAN = True     # needs pyjaspar and the hg38 genome (downloaded once)
GENOME_FASTA = None       # path to a local hg38 FASTA (.fa or .fa.gz); None downloads from UCSC
JASPAR_RELEASE = "JASPAR2024"
MOTIF_PVALUE = 1e-4       # per-window motif p-value threshold (the FIMO default)
MOTIF_TOP_PEAKS = 500     # strongest motif peaks kept per regulator, so set sizes are comparable
RUN_LSI_COMPARISON = True # train the paper's LSI model and compare perturbation effects
USE_GENCODE_TSS = False   # True: take TSS from GENCODE v32 instead of the coordinates stored in rna.var

## 1. Data and split

The Multiome RNA file stores gene coordinates in `.var` (`chrom`, `chromStart`, `chromEnd`,
`strand`), and the ATAC file stores peak coordinates. We use them to link regulators to peaks.

In [ ]:
data = uds.pbmc_multiome_10k()
rna, atac = data["rna"], data["atac"]
splits = split_by_label(rna.obs["cell_type"], train_fraction=0.8, val_fraction=0.1, seed=0)
print({k: len(v) for k, v in splits.items()})
print("RNA .var columns:", list(rna.var.columns)[:12], "...")
print("ATAC .var columns:", list(atac.var.columns))

### Gene TSS and peak coordinates

The TSS is `chromStart` for `+`-strand genes and `chromEnd` for `-`-strand genes. If the columns are
missing (or `USE_GENCODE_TSS = True`), gene coordinates come from the GENCODE v32 primary-assembly GTF,
the annotation of the 10x GRCh38-2020-A reference that Cell Ranger ARC uses. Print the table and check a
few genes you know before continuing.

In [ ]:
GENCODE_GTF = ("https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_32/"
               "gencode.v32.primary_assembly.annotation.gtf.gz")
CACHE = uds.get_data_dir() / "annotation"
CACHE.mkdir(parents=True, exist_ok=True)


def download(url, dest):
    dest = Path(dest)
    if not dest.exists():
        print(f"downloading {url}")
        tmp = dest.with_suffix(dest.suffix + ".part")
        urllib.request.urlretrieve(url, tmp)
        tmp.rename(dest)
    return dest


def tss_from_var(var):
    need = ["chrom", "chromStart", "chromEnd", "strand"]
    if not set(need) <= set(var.columns):
        return None
    v = var[need].dropna()
    v = v[v["strand"].astype(str).isin(["+", "-"])]
    if v.empty:
        return None
    tss = np.where(v["strand"].astype(str) == "-", v["chromEnd"], v["chromStart"]).astype(np.int64)
    return pd.DataFrame({"chrom": v["chrom"].astype(str).to_numpy(), "tss": tss,
                         "strand": v["strand"].astype(str).to_numpy()}, index=v.index)


def tss_from_gencode(url=GENCODE_GTF):
    path = download(url, CACHE / Path(url).name)
    rows = []
    with gzip.open(path, "rt") as fh:
        for line in fh:
            if line.startswith("#"):
                continue
            f = line.split("\t", 8)
            if f[2] != "gene":
                continue
            name = re.search(r'gene_name "([^"]+)"', f[8]).group(1)
            start, end = int(f[3]) - 1, int(f[4])          # GTF is 1-based; store 0-based starts
            rows.append((name, f[0], end if f[6] == "-" else start, f[6]))
    df = pd.DataFrame(rows, columns=["gene", "chrom", "tss", "strand"])
    return df.drop_duplicates("gene").set_index("gene")


def peak_table(adata):
    if {"chrom", "chromStart", "chromEnd"} <= set(adata.var.columns):
        v = adata.var
        return pd.DataFrame({"chrom": v["chrom"].astype(str).to_numpy(),
                             "start": v["chromStart"].astype(np.int64).to_numpy(),
                             "end": v["chromEnd"].astype(np.int64).to_numpy()}, index=adata.var_names)
    parts = adata.var_names.to_series().str.extract(r"^(.+?)[:\-_](\d+)[\-_](\d+)$")
    if parts.isna().any().any():
        raise ValueError("Could not parse peak coordinates from var_names.")
    return pd.DataFrame({"chrom": parts[0].to_numpy(), "start": parts[1].astype(np.int64).to_numpy(),
                         "end": parts[2].astype(np.int64).to_numpy()}, index=adata.var_names)


tss = None if USE_GENCODE_TSS else tss_from_var(rna.var)
tss_source = "rna.var"
if tss is None:
    tss, tss_source = tss_from_gencode(), "GENCODE v32"
peaks_all = peak_table(atac)
print(f"TSS for {len(tss):,} genes from {tss_source}; {len(peaks_all):,} peaks")
shared_chroms = set(tss["chrom"]) & set(peaks_all["chrom"])
assert shared_chroms, "Gene and peak chromosome names do not match (e.g. 'chr1' vs '1')."

panel = [g for g in TF_PANEL if g in rna.var_names and g in tss.index]
print("regulators used:", panel, "| dropped:", sorted(set(TF_PANEL) - set(panel)))
tss.loc[panel]

## 2. Features: genes and peaks

RNA features are the training-set HVGs plus every panel regulator, kept as log-normalized expression
(not z-scored), so predicted changes read in log1p(CP10k) units. ATAC features are the `N_PEAKS` peaks
open in the most training cells, plus every peak within `CIS_WINDOW_BP` of a regulator's TSS, binarized
(open = 1). Everything is fit on training cells only.

In [ ]:
def peaks_near(tss_row, peaks, window):
    same = peaks["chrom"].to_numpy() == tss_row["chrom"]
    mid = (peaks["start"].to_numpy() + peaks["end"].to_numpy()) // 2
    return peaks.index[same & (np.abs(mid - int(tss_row["tss"])) <= window)]


hvg = RNAPreprocessor(n_hvg=N_HVG).fit(rna[splits["train"]]).features_
genes = list(dict.fromkeys(hvg + panel))
rna_prep = RNAPreprocessor(n_hvg=None, scale=False).fit(rna[splits["train"]][:, genes])

train_counts = atac[splits["train"]].layers["counts"]
peak_freq = pd.Series(np.asarray((train_counts > 0).mean(axis=0)).ravel(), index=atac.var_names)
cis_all = {tf: peaks_near(tss.loc[tf], peaks_all, CIS_WINDOW_BP) for tf in panel}
top_peaks = peak_freq[peak_freq > 0].sort_values(ascending=False).index[:N_PEAKS]
model_peaks = list(dict.fromkeys(list(top_peaks) + [p for ps in cis_all.values() for p in ps
                                                    if peak_freq[p] > 0]))
peak_idx = atac.var_names.get_indexer(model_peaks)


def binarize(adata_counts):
    x = sp.csr_matrix(adata_counts.layers["counts"][:, peak_idx], dtype=np.float32)
    x.data = np.ones_like(x.data)
    return ad.AnnData(x, obs=adata_counts.obs.copy(), var=peaks_all.loc[model_peaks].copy())


parts = {k: {"rna": rna_prep.transform(rna[i]), "atac": binarize(atac[i])} for k, i in splits.items()}
train, val, test = parts["train"], parts["val"], parts["test"]
print("RNA", train["rna"].shape, "| ATAC", train["atac"].shape)

## 3. Train the peak-level model

Settings follow the quickstart. `ATAC_RECON_WEIGHT` scales the Bernoulli term, which sums over many more
features than the RNA term; 0.1 is a starting value, not a tuned one.

In [ ]:
cfg = UniVIConfig(
    latent_dim=30, beta=1.25, gamma=4.35, encoder_dropout=0.10, decoder_dropout=0.05,
    kl_anneal_start=50, kl_anneal_end=85, align_anneal_start=75, align_anneal_end=110,
    modalities=[
        ModalityConfig("rna", train["rna"].n_vars, [512, 256, 128], [128, 256, 512], likelihood="gaussian"),
        ModalityConfig("atac", train["atac"].n_vars, [512, 256, 128], [128, 256, 512], likelihood="bernoulli",
                       recon_weight=ATAC_RECON_WEIGHT),
    ],
)
model = UniVIMultiModalVAE(cfg, loss_mode="v1", v1_recon="avg", normalize_v1_terms=True)
train_cfg = TrainingConfig(n_epochs=N_EPOCHS, batch_size=BATCH_SIZE, lr=1e-3, weight_decay=1e-4, device=device,
                           early_stopping=True, patience=50, best_epoch_warmup=110, log_every=50)
trainer = UniVITrainer(model, make_loader(train, batch_size=BATCH_SIZE, shuffle=True, drop_last=True),
                       make_loader(val, batch_size=1024), train_cfg)
history = trainer.fit()
print("best epoch:", trainer.best_epoch)

## 4. Gate the model before perturbing it

A perturbation readout can only be as good as the model's ATAC → RNA mapping. We check paired-cell
alignment on held-out cells, then per-gene accuracy of RNA predicted from ATAC alone. The regulators'
own prediction accuracy is printed separately: if a regulator's expression is poorly predicted from
chromatin, its perturbation results deserve little weight.

In [ ]:
z_rna = encode_adata(model, test["rna"], modality="rna", device=device, latent="modality_mean")
z_atac = encode_adata(model, test["atac"], modality="atac", device=device, latent="modality_mean")
labels = test["rna"].obs["cell_type"].astype(str).to_numpy()
m = evaluate_alignment(Z1=z_rna, Z2=z_atac, labels_source=labels, labels_target=labels, recall_ks=(1, 10))
print(pd.Series({"FOSCTTM (lower is better)": m["foscttm_mean"],
                 "Recall@10": m["recall_at_k"]["10"]["mean"],
                 "label transfer RNA->ATAC": m["label_transfer_acc"]}).round(3))

baseline = cross_modal_predict(model, test["atac"], src_mod="atac", tgt_mod="rna", device=device)
observed = dense(test["rna"].X)
gene_r = pd.Series(pearson_corr_per_feature(observed, baseline), index=test["rna"].var_names)

fig, ax = plt.subplots(figsize=(5, 3))
ax.hist(gene_r.dropna(), bins=50, color="0.6")
for tf in panel:
    ax.axvline(gene_r[tf], lw=0.8, color="C3")
ax.set(xlabel="Pearson r, observed vs predicted from ATAC", ylabel="genes",
       title=f"ATAC → RNA on test cells (median r = {gene_r.median():.2f}; red = regulators)")
plt.show()
gate = gene_r.loc[panel].rename("pearson_r_from_atac").to_frame()
gate["expressed_in_top_type"] = [
    pd.Series(observed[:, test["rna"].var_names.get_loc(tf)], index=labels).groupby(level=0).mean().idxmax()
    for tf in panel]
gate.round(3)

A shared latent embedding of the test cells is used for plotting below.

In [ ]:
test_rna = test["rna"]
test_rna.obsm["X_univi"] = z_rna
sc.pp.neighbors(test_rna, use_rep="X_univi", n_neighbors=30)
sc.tl.umap(test_rna, random_state=0)
sc.pl.umap(test_rna, color="cell_type", legend_fontsize=7, title="test cells (RNA latent)")

## 5. Peak sets for each regulator

### Cis: peaks around the regulator's own gene

Closing the chromatin around a gene is the most direct perturbation the model can see. The expected
first-order readout is a drop in that gene's predicted expression, in the cell types that express it.

In [ ]:
peak_sets = {}
for tf in panel:
    cis = [p for p in cis_all[tf] if p in set(model_peaks)]
    if cis:
        peak_sets[(tf, "cis")] = cis
pd.Series({tf: len(peak_sets.get((tf, "cis"), [])) for tf in panel}, name="cis peaks in model").to_frame().T

### Motif: peaks carrying the regulator's binding motif (optional)

Each model peak is scanned on both strands with the regulator's JASPAR position weight matrix, and its
best window is converted to a **p-value**: the probability that a random window scores at least as
high, computed exactly from the matrix under the base composition of the scanned peaks. P-values are
comparable across matrices of different widths, which relative scores are not (a short matrix reaches
a high relative score by chance far more often than a long one). Peaks with p ≤ `MOTIF_PVALUE` qualify,
and the `MOTIF_TOP_PEAKS` strongest are kept so that every regulator gets a set of similar size.
A peak of length L has about 2L windows, so at p = 1e-4 a 500-bp peak contains a chance match about
10% of the time; keeping only the strongest matches reduces, but does not remove, such hits.

A motif match is a weak proxy for binding: members of a family share motifs (for example ETS factors
with SPI1, T-box factors TBX21 and EOMES, C/EBP family members), so a "motif" perturbation is better
read as "sites of this motif family".

In [ ]:
def load_log_odds(tfs, release=JASPAR_RELEASE, pseudocount=0.8):
    from pyjaspar import jaspardb
    jdb = jaspardb(release=release)
    out = {}
    for tf in tfs:
        motifs = jdb.fetch_motifs_by_name(tf)
        if not motifs:
            print(f"  no {release} motif named {tf}; skipped")
            continue
        m = motifs[0]
        counts = np.array([m.counts[b] for b in "ACGT"], dtype=float).T          # (width, 4)
        probs = (counts + pseudocount / 4) / (counts.sum(1, keepdims=True) + pseudocount)
        out[tf] = (m.matrix_id, np.log2(probs / 0.25))
    return out


def iter_fasta(path, keep):
    """Yield (chromosome, uppercase sequence bytes) for the chromosomes in `keep`, one at a time."""
    opener = gzip.open if str(path).endswith(".gz") else open
    name, chunks = None, []
    with opener(path, "rb") as fh:
        for line in fh:
            if line.startswith(b">"):
                if name in keep:
                    yield name, b"".join(chunks).upper()
                name, chunks = line[1:].split()[0].decode(), []
            elif name in keep:
                chunks.append(line.rstrip())
    if name in keep:
        yield name, b"".join(chunks).upper()


LOOKUP = np.full(256, 4, dtype=np.int8)
for i, b in enumerate(b"ACGT"):
    LOOKUP[b] = i


def best_scores(seq_codes, pwms):
    """Best log-odds window score (both strands) of one sequence per motif; windows with N never win."""
    out = {}
    for tf, (lo, lo_rc) in pwms.items():
        w = lo.shape[0]
        if len(seq_codes) < w:
            out[tf] = -np.inf
            continue
        win = np.lib.stride_tricks.sliding_window_view(seq_codes, w)
        cols = np.arange(w)
        out[tf] = max(lo[cols, win].sum(1).max(), lo_rc[cols, win].sum(1).max())
    return out


def score_tail(lo, background, resolution=100):
    """Exact P(window score >= s) under an i.i.d. background, via dynamic programming on rounded scores."""
    q = np.round(lo * resolution).astype(np.int64)
    offset = q.min(1)
    dist = np.array([1.0])
    for row, low in zip(q, offset):
        new = np.zeros(len(dist) + int((row - low).max()))
        for base in range(4):
            k = int(row[base] - low)
            new[k:k + len(dist)] += background[base] * dist
        dist = new
    tail = np.cumsum(dist[::-1])[::-1]
    return lambda s: tail[np.clip(np.ceil(np.asarray(s) * resolution - 1e-9).astype(np.int64) - offset.sum(),
                                  0, len(tail) - 1)] * (np.asarray(s) > -np.inf)


def scan_peaks(peaks, log_odds, fasta):
    """Best motif p-value per peak and regulator (background = base composition of the scanned peaks)."""
    pwms = {}
    for tf, (_, lo) in log_odds.items():
        pwms[tf] = (np.hstack([lo, np.full((lo.shape[0], 1), -1e3)]),            # column 4: N
                    np.hstack([lo[::-1, ::-1], np.full((lo.shape[0], 1), -1e3)]))  # reverse complement
    tfs = list(pwms)
    best = np.full((len(peaks), len(tfs)), -np.inf)
    base_counts = np.zeros(5)
    pos = pd.Series(np.arange(len(peaks)), index=peaks.index)
    by_chrom = {c: df for c, df in peaks.groupby("chrom")}
    seen = set()
    for chrom, seq in iter_fasta(fasta, set(by_chrom)):
        codes = LOOKUP[np.frombuffer(seq, dtype=np.uint8)]
        df = by_chrom[chrom]
        for row_i, start, end in zip(pos[df.index].to_numpy(), df["start"].to_numpy(), df["end"].to_numpy()):
            window = codes[start:end]
            base_counts += np.bincount(window, minlength=5)
            b = best_scores(window, pwms)
            best[row_i] = [b[tf] for tf in tfs]
        seen.add(chrom)
    missing = set(by_chrom) - seen
    if missing:
        print(f"  chromosomes not in the FASTA (their peaks get p = 1): {sorted(missing)}")
    background = base_counts[:4] / base_counts[:4].sum()
    print("  peak base composition (A, C, G, T):", np.round(background, 3))
    pvals = np.ones_like(best)
    for j, tf in enumerate(tfs):
        pvals[:, j] = np.where(np.isfinite(best[:, j]), score_tail(log_odds[tf][1], background)(best[:, j]), 1.0)
    return pd.DataFrame(pvals, index=peaks.index, columns=tfs)


motif_pvals = None
if RUN_MOTIF_SCAN:
    log_odds = load_log_odds(panel)
    print({tf: mid for tf, (mid, _) in log_odds.items()})
    fasta = GENOME_FASTA or download("https://hgdownload.soe.ucsc.edu/goldenPath/hg38/bigZips/hg38.fa.gz",
                                     CACHE / "hg38.fa.gz")
    motif_pvals = scan_peaks(peaks_all.loc[model_peaks], log_odds, fasta)
    for tf in motif_pvals.columns:
        pv = motif_pvals[tf]
        chosen = pv[pv <= MOTIF_PVALUE].sort_values(kind="stable").index[:MOTIF_TOP_PEAKS]
        print(f"{tf}: {int((pv <= MOTIF_PVALUE).sum()):,} of {len(pv):,} peaks have a match at "
              f"p <= {MOTIF_PVALUE:g}; keeping {len(chosen)}")
        if len(chosen):
            peak_sets[(tf, "motif")] = list(chosen)

pd.Series({f"{tf} ({kind})": len(p) for (tf, kind), p in peak_sets.items()}, name="peaks").to_frame().T

## 6. Close the peaks and decode RNA

`predict_feature_perturbation` edits the named ATAC inputs (here: set to 0, i.e. closed), runs the
original and edited inputs through the ATAC encoder and the RNA decoder, and returns both predictions
and their difference. Peaks that were already closed in a cell do not change that cell's input, so the
per-cell effect depends on how many of the set's peaks were open.

In [ ]:
res = predict_feature_perturbation(model, test["atac"], source_modality="atac", target_modality="rna",
                                   features=peak_sets[next(iter(peak_sets))], mode="off", device=device)
assert peak_sets, "No peak sets were built; check the TSS/peak coordinate tables above."
assert np.allclose(res["baseline"], baseline, atol=1e-5)   # same inputs, same model: same baseline
cell_types = pd.Index(sorted(set(labels)))


def by_type(delta):
    """Mean predicted change per cell type (rows) and gene (columns)."""
    return pd.DataFrame(delta, index=labels, columns=test_rna.var_names).groupby(level=0).mean().loc[cell_types]


effects, deltas = {}, {}
for key, peaks in peak_sets.items():
    out = predict_feature_perturbation(model, test["atac"], source_modality="atac", target_modality="rna",
                                       features=peaks, mode="off", device=device)
    deltas[key] = out["delta"]
    effects[key] = by_type(out["delta"])
print(f"{len(effects)} perturbations; each is a cell types x genes table of mean predicted change")

## 7. Empirical null: random peak sets matched for accessibility

Closing any few hundred open peaks moves the latent a little, and some genes move with almost any
latent shift. To separate regulator-specific effects from that, each perturbation is compared with
`N_NULL` random peak sets of the same size, drawn to match the real set's distribution of training-cell
accessibility (ten frequency bins). The null is computed with a small helper that zeroes columns and
decodes; the first line checks that it reproduces `predict_feature_perturbation` exactly.

In [ ]:
X_test = sp.csr_matrix(test["atac"].X)
freq_model = peak_freq.loc[model_peaks].to_numpy()
freq_bin = pd.qcut(pd.Series(freq_model).rank(method="first"), 10, labels=False).to_numpy()
name_to_col = pd.Series(np.arange(len(model_peaks)), index=model_peaks)


def closed_delta(cols):
    mask = np.ones(X_test.shape[1], dtype=np.float32)
    mask[cols] = 0.0
    edited = ad.AnnData(sp.csr_matrix(X_test.multiply(mask[None, :])), obs=test["atac"].obs)
    return cross_modal_predict(model, edited, src_mod="atac", tgt_mod="rna", device=device) - baseline


first = next(iter(peak_sets))
assert np.allclose(closed_delta(name_to_col[peak_sets[first]].to_numpy()), deltas[first], atol=1e-5)


def matched_random_sets(cols, n, rng):
    cols = np.asarray(cols)
    need = pd.Series(freq_bin[cols]).value_counts()
    pool = {b: np.setdiff1d(np.flatnonzero(freq_bin == b), cols) for b in need.index}
    return [np.concatenate([rng.choice(pool[b], size=int(c), replace=False) for b, c in need.items()])
            for _ in range(n)]


rng = np.random.default_rng(0)
null_mean, null_sd, zscores, null_self = {}, {}, {}, {}
for key, peaks in peak_sets.items():
    draws = np.stack([by_type(closed_delta(c)).to_numpy()
                      for c in matched_random_sets(name_to_col[peaks].to_numpy(), N_NULL, rng)])
    null_mean[key] = pd.DataFrame(draws.mean(0), index=cell_types, columns=test_rna.var_names)
    null_sd[key] = pd.DataFrame(draws.std(0, ddof=1), index=cell_types, columns=test_rna.var_names)
    zscores[key] = (effects[key] - null_mean[key]) / (null_sd[key] + 1e-8)
    null_self[key] = pd.DataFrame(draws[:, :, test_rna.var_names.get_loc(key[0])], columns=cell_types)
    print(f"{key}: null done")

The z-score of an effect is `(observed - null mean) / null s.d.` for each cell type and gene. Treat it
as a ranking statistic: with `N_NULL = 50` the null is estimated from 50 draws, and converting z to a
p-value assumes the null is roughly Gaussian. The next plot shows the null for each regulator's own gene
in the cell type that expresses it most, with the observed effect as a red line.

In [ ]:
keys = list(peak_sets)
ncol = min(5, len(keys))
fig, axes = plt.subplots(int(np.ceil(len(keys) / ncol)), ncol, figsize=(2.6 * ncol, 2.3 * np.ceil(len(keys) / ncol)),
                         squeeze=False)
for ax, key in zip(axes.ravel(), keys):
    tf, ct = key[0], gate.loc[key[0], "expressed_in_top_type"]
    ax.hist(null_self[key][ct], bins=15, color="0.7")
    ax.axvline(effects[key].loc[ct, tf], color="C3")
    ax.set_title(f"{tf} ({key[1]}) in {ct}", fontsize=7)
    ax.set_xlabel(f"Δ {tf} (log1p)", fontsize=7)
for ax in axes.ravel()[len(keys):]:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 8. What moved

### The regulator's own expression

Rows are perturbations, columns are cell types; color is the z-score of the change in the regulator's
own predicted expression. For cis sets, a negative value where the regulator is expressed is the
expected direction.

In [ ]:
self_z = pd.DataFrame({f"{tf} ({kind})": zscores[(tf, kind)][tf] for (tf, kind) in peak_sets}).T
fig, ax = plt.subplots(figsize=(0.45 * len(cell_types) + 2, 0.35 * len(self_z) + 1.5))
lim = np.nanmax(np.abs(self_z.to_numpy())) or 1.0
im = ax.imshow(self_z.to_numpy(), cmap="RdBu_r", vmin=-lim, vmax=lim, aspect="auto")
ax.set_xticks(range(len(cell_types)), cell_types, rotation=90, fontsize=7)
ax.set_yticks(range(len(self_z)), self_z.index, fontsize=8)
plt.colorbar(im, ax=ax, label="z vs matched random peaks")
ax.set_title("Change in the regulator's own predicted expression")
plt.tight_layout()
plt.show()

### Genes that respond beyond the null

For each perturbation we rank genes by their most extreme z-score across cell types and show the mean
predicted change of the top genes in every cell type. Genes are the rows; dots mark |z| ≥ 3.

In [ ]:
def top_genes(key, n=15):
    z = zscores[key]
    score = z.abs().max(axis=0).sort_values(ascending=False)
    return score.index[:n]


def plot_response(key, n=15):
    g = top_genes(key, n)
    eff, z = effects[key][g].T, zscores[key][g].T
    fig, ax = plt.subplots(figsize=(0.4 * len(cell_types) + 2.5, 0.28 * len(g) + 1.6))
    lim = np.abs(eff.to_numpy()).max() or 1.0
    im = ax.imshow(eff.to_numpy(), cmap="RdBu_r", vmin=-lim, vmax=lim, aspect="auto")
    yy, xx = np.nonzero(np.abs(z.to_numpy()) >= 3)
    ax.scatter(xx, yy, s=6, c="k")
    ax.set_xticks(range(len(cell_types)), cell_types, rotation=90, fontsize=7)
    ax.set_yticks(range(len(g)), g, fontsize=7)
    plt.colorbar(im, ax=ax, label="mean Δ (log1p)")
    ax.set_title(f"Closing {key[0]} {key[1]} peaks ({len(peak_sets[key])} peaks)", fontsize=9)
    plt.tight_layout()
    plt.show()


for key in peak_sets:
    plot_response(key)

### Effect size against evidence

One panel per perturbation, in the cell type that expresses the regulator most: mean predicted change
against |z|. Genes in the top right or top left are both large and unusual relative to random closures.

In [ ]:
keys = list(peak_sets)
ncol = min(4, len(keys))
nrow = int(np.ceil(len(keys) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(3.2 * ncol, 2.8 * nrow), squeeze=False)
for ax, key in zip(axes.ravel(), keys):
    ct = gate.loc[key[0], "expressed_in_top_type"]
    x, y = effects[key].loc[ct], zscores[key].loc[ct].abs()
    ax.scatter(x, y, s=4, c="0.6")
    for gname in y.sort_values(ascending=False).index[:6]:
        ax.annotate(gname, (x[gname], y[gname]), fontsize=6)
    ax.axhline(3, lw=0.5, ls="--", c="k")
    ax.set_title(f"{key[0]} ({key[1]}) in {ct}", fontsize=8)
    ax.set_xlabel("mean Δ (log1p)", fontsize=7)
    ax.set_ylabel("|z|", fontsize=7)
for ax in axes.ravel()[len(keys):]:
    ax.axis("off")
plt.tight_layout()
plt.show()

### Per-cell view

The mean hides which cells respond. Coloring the test-cell UMAP by the per-cell change in the
regulator's own predicted expression shows where the model places the response.

In [ ]:
show = [k for k in peak_sets][:4]
for key in show:
    test_rna.obs[f"Δ{key[0]} ({key[1]})"] = deltas[key][:, test_rna.var_names.get_loc(key[0])]
sc.pl.umap(test_rna, color=[f"Δ{k[0]} ({k[1]})" for k in show], cmap="RdBu_r", vcenter=0, ncols=4)

## 9. Where do perturbed cells move in the latent space?

Encoding the edited ATAC places each perturbed cell in the latent space. Two readouts:

- **displacement**: distance between a cell's original and perturbed embedding, averaged per cell
  type, next to the same quantity for one matched random closure (`excess_over_random` is the
  difference);
- **label shift**: the fraction of cells whose nearest-neighbor cell-type label (voted by the RNA
  embeddings of the test cells) changes after the edit, and what they change to.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=15).fit(z_rna, labels)   # labeled reference: RNA embeddings
before = knn.predict(z_atac)


def encode_closed(cols):
    mask = np.ones(X_test.shape[1], dtype=np.float32)
    mask[cols] = 0.0
    edited = ad.AnnData(sp.csr_matrix(X_test.multiply(mask[None, :])), obs=test["atac"].obs)
    return encode_adata(model, edited, modality="atac", device=device, latent="modality_mean")


shift_rows, transitions, changed = [], {}, {}
rng_shift = np.random.default_rng(2)
for key, peaks in peak_sets.items():
    cols = name_to_col[peaks].to_numpy()
    z_pert = encode_closed(cols)
    z_null = encode_closed(matched_random_sets(cols, 1, rng_shift)[0])
    disp = pd.Series(np.linalg.norm(z_pert - z_atac, axis=1), index=labels).groupby(level=0).mean()
    disp_null = pd.Series(np.linalg.norm(z_null - z_atac, axis=1), index=labels).groupby(level=0).mean()
    after = knn.predict(z_pert)
    transitions[key] = pd.crosstab(pd.Series(before, name="before"), pd.Series(after, name="after"))
    changed[key] = float((before != after).mean())
    shift_rows.append(pd.DataFrame({"perturbation": f"{key[0]} ({key[1]})", "cell_type": disp.index,
                                    "displacement": disp.values, "random_set": disp_null.values,
                                    "label_changed": pd.Series(before != after, index=labels)
                                    .groupby(level=0).mean().loc[disp.index].values}))
shift = pd.concat(shift_rows, ignore_index=True)
shift["excess_over_random"] = shift["displacement"] - shift["random_set"]
shift.sort_values("excess_over_random", ascending=False).head(15).round(3)

In [ ]:
print({f"{k[0]} ({k[1]})": round(v, 3) for k, v in changed.items()})
key = max(changed, key=changed.get)   # the perturbation that changes the most labels
t = transitions[key]
t = t.div(t.sum(axis=1), axis=0)
fig, ax = plt.subplots(figsize=(0.35 * t.shape[1] + 2.5, 0.3 * t.shape[0] + 1.5))
im = ax.imshow(t.to_numpy(), cmap="viridis", vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(t.shape[1]), t.columns, rotation=90, fontsize=7)
ax.set_yticks(range(t.shape[0]), t.index, fontsize=7)
ax.set(xlabel="nearest-neighbor label after", ylabel="label before")
ax.set_title(f"Label shifts after closing {key[0]} ({key[1]}) peaks", fontsize=9)
plt.colorbar(im, ax=ax, label="fraction of cells")
plt.tight_layout()
plt.show()

## 10. Dose-response

Bernoulli inputs must stay binary, so "dose" here is the fraction of a peak set that is closed. For the
largest set, peaks are closed in order (strongest motif first for motif sets, nearest the TSS first for
cis sets), and the response of the regulator's own gene and of the perturbation's top responders is
tracked in the most relevant cell type.

In [ ]:
key = max(peak_sets, key=lambda k: len(peak_sets[k]))
tf, kind = key
ordered = list(peak_sets[key])
if kind == "motif" and motif_pvals is not None:
    ordered = list(motif_pvals.loc[ordered, tf].sort_values(kind="stable").index)
else:
    mid = (peaks_all.loc[ordered, "start"] + peaks_all.loc[ordered, "end"]) // 2
    ordered = list((mid - tss.loc[tf, "tss"]).abs().sort_values().index)
ct = gate.loc[tf, "expressed_in_top_type"]
track = list(dict.fromkeys([tf] + list(top_genes(key, 5))))
fractions = [0.1, 0.25, 0.5, 0.75, 1.0]
dose = pd.DataFrame({f: by_type(closed_delta(name_to_col[ordered[:max(1, int(f * len(ordered)))]]
                                                 .to_numpy())).loc[ct, track] for f in fractions}).T
ax = dose.plot(marker="o", figsize=(5, 3.2))
ax.axhline(0, c="k", lw=0.5)
ax.set(xlabel="fraction of peak set closed", ylabel=f"mean Δ in {ct} (log1p)",
       title=f"Dose-response: {tf} ({kind}), {len(ordered)} peaks")
ax.legend(fontsize=7, frameon=False, bbox_to_anchor=(1, 1))
plt.show()

## 11. Two regulators at once

Closing the sets of two regulators together and comparing with the sum of the single closures gives an
**interaction** term, `Δ(A+B) − Δ(A) − Δ(B)`. Zero means the model combines the two edits additively;
a large value means it does not. The pair used is the first two motif sets if available, otherwise
the first two sets.

In [ ]:
motif_keys = [k for k in peak_sets if k[1] == "motif"]
pair = motif_keys[:2] if len(motif_keys) >= 2 else list(peak_sets)[:2]
if len(pair) == 2:
    a, b = pair
    both = np.union1d(name_to_col[peak_sets[a]].to_numpy(), name_to_col[peak_sets[b]].to_numpy())
    combo = by_type(closed_delta(both))
    interaction = combo - effects[a] - effects[b]
    genes_show = list(dict.fromkeys(list(top_genes(a, 6)) + list(top_genes(b, 6))))
    ct = gate.loc[a[0], "expressed_in_top_type"]
    summary = pd.DataFrame({f"{a[0]} alone": effects[a].loc[ct, genes_show],
                            f"{b[0]} alone": effects[b].loc[ct, genes_show],
                            "both": combo.loc[ct, genes_show],
                            "interaction": interaction.loc[ct, genes_show]})
    print(f"{a} + {b}, in {ct}; median |interaction| over all genes and cell types: "
          f"{np.median(np.abs(interaction.to_numpy())):.2e}")
    display(summary.round(4))

## 12. Does an independently trained model agree?

A result that depends on one model's idiosyncrasies should not replicate in a model with a different
ATAC representation. Here we train the paper's Multiome setup (ATAC as TF-IDF/LSI over **all** peaks,
Gaussian likelihood) on the same cells and RNA features, apply the same closures to the raw peak counts,
re-run the fitted TF-IDF/LSI transform, and decode RNA. For each perturbation we report the Spearman
correlation between the two models' per-gene effects in the relevant cell type.

In [ ]:
if RUN_LSI_COMPARISON:
    from scipy.stats import spearmanr

    atac_prep = ATACPreprocessor(n_components=N_LSI, drop_first=True, scale=True).fit(atac[splits["train"]])
    lsi = {k: atac_prep.transform(atac[i]) for k, i in splits.items()}
    cfg_lsi = UniVIConfig(
        latent_dim=30, beta=1.25, gamma=4.35, encoder_dropout=0.10, decoder_dropout=0.05,
        kl_anneal_start=50, kl_anneal_end=85, align_anneal_start=75, align_anneal_end=110,
        modalities=[ModalityConfig("rna", train["rna"].n_vars, [512, 256, 128], [128, 256, 512]),
                    ModalityConfig("atac", lsi["train"].n_vars, [128, 64], [64, 128])])
    model_lsi = UniVIMultiModalVAE(cfg_lsi, loss_mode="v1", v1_recon="avg", normalize_v1_terms=True)
    UniVITrainer(model_lsi, make_loader({"rna": train["rna"], "atac": lsi["train"]}, batch_size=BATCH_SIZE,
                                        shuffle=True, drop_last=True),
                 make_loader({"rna": val["rna"], "atac": lsi["val"]}, batch_size=1024), train_cfg).fit()

    atac_test_raw = atac[splits["test"]].copy()
    base_lsi = cross_modal_predict(model_lsi, lsi["test"], src_mod="atac", tgt_mod="rna", device=device)
    r_lsi = pd.Series(pearson_corr_per_feature(observed, base_lsi), index=test_rna.var_names)
    print(f"LSI model, ATAC -> RNA median per-gene r: {r_lsi.median():.3f} "
          f"(peak-level model: {gene_r.median():.3f})")

    def closed_delta_lsi(peaks):
        edited = atac_test_raw.copy()
        counts = sp.csr_matrix(edited.layers["counts"], dtype=np.float32)
        mask = np.ones(counts.shape[1], dtype=np.float32)
        mask[edited.var_names.get_indexer(peaks)] = 0.0
        edited.layers["counts"] = sp.csr_matrix(counts.multiply(mask[None, :]))
        pred = cross_modal_predict(model_lsi, atac_prep.transform(edited), src_mod="atac", tgt_mod="rna",
                                   device=device)
        return pred - base_lsi

    rows = []
    effects_lsi = {}
    for key, peaks in peak_sets.items():
        effects_lsi[key] = by_type(closed_delta_lsi(peaks))
        ct = gate.loc[key[0], "expressed_in_top_type"]
        rho = spearmanr(effects[key].loc[ct], effects_lsi[key].loc[ct]).statistic
        rows.append({"perturbation": f"{key[0]} ({key[1]})", "cell_type": ct, "spearman_rho": rho,
                     f"Δ{key[0]} peak-level": effects[key].loc[ct, key[0]],
                     f"Δ{key[0]} LSI": effects_lsi[key].loc[ct, key[0]]})
    agreement = pd.DataFrame(rows).set_index("perturbation")
    display(agreement[["cell_type", "spearman_rho"]].round(3))

    key = agreement["spearman_rho"].idxmax() if agreement["spearman_rho"].notna().any() else None
    if key is not None:
        k = next(k for k in peak_sets if f"{k[0]} ({k[1]})" == key)
        ct = gate.loc[k[0], "expressed_in_top_type"]
        fig, ax = plt.subplots(figsize=(3.6, 3.4))
        ax.scatter(effects[k].loc[ct], effects_lsi[k].loc[ct], s=4, c="0.5")
        ax.axhline(0, lw=0.5, c="k")
        ax.axvline(0, lw=0.5, c="k")
        ax.set(xlabel="Δ peak-level model", ylabel="Δ LSI model", title=f"{key} in {ct}")
        plt.show()

## 13. Save the results

One long table with every perturbation, cell type and gene: observed mean change, null mean and s.d.,
and z-score. Filter it rather than re-running the notebook.

In [ ]:
long = []
for key in peak_sets:
    df = pd.DataFrame({"delta": effects[key].stack(), "null_mean": null_mean[key].stack(),
                       "null_sd": null_sd[key].stack(), "z": zscores[key].stack()})
    df.index.names = ["cell_type", "gene"]
    df = df.reset_index()
    df.insert(0, "set", key[1])
    df.insert(0, "regulator", key[0])
    df["n_peaks"] = len(peak_sets[key])
    long.append(df)
results = pd.concat(long, ignore_index=True)
results.to_csv("tf_perturbation_effects.csv.gz", index=False)
results.reindex(results["z"].abs().sort_values(ascending=False).index).head(20).round(4)

## Reading the results

- **Start from the gate.** Regulators whose expression is poorly predicted from ATAC (Section 4) cannot
  give informative cis readouts, whatever their z-scores.
- **Prefer effects that pass the null and replicate.** An effect with large |z| that also has the same
  sign in the LSI model (Section 12) is a better lead than one that appears in a single model. Training a
  second seed of the peak-level model is an inexpensive further check.
- **Motif sets are family-level.** Overlapping motifs mean that a "SPI1 motif" perturbation also removes
  sites of related ETS factors.
- **The model knows only co-variation.** It has learned which accessibility patterns accompany which
  expression patterns across cell types. Closing a set of peaks can therefore move a cell toward another
  cell type's profile (Section 9) without that being what a real perturbation would do.

Ideas to extend this notebook: use cell-type-specific peak subsets, perturb with `mode="set", value=1`
(opening peaks rather than closing them), or run the same analysis on the TEA-seq data to read out
predicted surface protein as well as RNA.